# Simple RAG Demo ---> HR Policy Assistant

**RAG (Retrieval-Augmented Generation)** is a technique that combines information retrieval with a language model to generate accurate, context-aware responses.

In this demo:

1. We use an HR policy document as the knowledge source.
2. The document is split into smaller chunks for efficient retrieval.
3. Each chunk is converted into an embedding (a numerical representation of its meaning).
4. These embeddings are stored in a FAISS vector database.
5. When a user asks a question, the system retrieves the most relevant chunks from the vector store.
6. The retrieved context is provided to a language model, which generates a clear and accurate response.

In simple terms, RAG combines **semantic search** with a **large language model** to answer questions based on the contents of a document.

## Technologies Used

* **Data:** `data/hr_policy.txt` (sample HR policy document)
* **Embeddings:** Jina AI
* **Vector Store:** FAISS
* **Language Model (LLM):** Groq
* **Framework:** LangChain (`create_agent`)

## Prerequisites

Before running this notebook:

* Ensure the notebook is using the **`ragenv`** virtual environment (select it from the top-right corner in VS Code or Jupyter).
* Create a `.env` file in the project directory containing the following API keys:

  * `GROQ_API_KEY`
  * `JINA_API_KEY`


###  Import everything we need
 We import all the tools upfront so its clear whats being used and where it comes from

In [32]:
import os
from dotenv import load_dotenv

#langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


In [3]:
load_dotenv()

True

In [6]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")


ENV VAR LOADED


### LOADING THE DATA

In [7]:
DATA_FILE_PATH = os.path.join('data', 'hr_policy.txt')

### DATA INGESTION

In [9]:
loader = TextLoader(DATA_FILE_PATH,  encoding ='utf-8')

documents = loader.load()

print("Data loaded")
print("="*50)
print(documents)
print("="*50)

Data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

### LANCHIAN DOCUMENT

Langchain processes everything in form of documents

DOCUMENTS:
PAGE CONTENT --- the actual data
METADATA --- extra info about the data


In [10]:
len(documents)

1

In [13]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [15]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [16]:
print(f'Total characters in document : {len(documents[0].page_content)}')

Total characters in document : 2597


### SPLITTING DATA

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='Sick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nE

In [21]:
len(chunks)

9

In [25]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


In [ ]:
print(chunks[5].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [30]:
print(chunks[6].page_content)

4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final settlement
processed within 45 days of the last working day.


### EMBEDD OUR DATA

In [35]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name ="jina-embeddings-v2-base-en")

print('Embedding model: ', embeddings_model.model_name)


Embedding model:  jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [36]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks , embeddings_model)
print("Chunks are stored", vector_store.index.ntotal)

Chunks are stored 14


In [37]:
# SIMILARITY SEARCH

test_query = "How many sick leaves employees get"

top_matches  = vector_store.similarity_search(test_query, k=2)
print(f"Query:{test_query}")
for i, match in enumerate(top_matches, start =1):
    print(f'-----Match {i} ------')
    print(match.page_content)
    print()

Query:How many sick leaves employees get
-----Match 1 ------
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

-----Match 2 ------
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### TOOL


In [48]:
retriever = vector_store.as_retriever(search_kwargs ={'k':3}) # retrurns top 3 relevant chunks

def search_hr_policy(question:str) -> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    """
    matching_chunks = retriever.invoke(question)
    
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)
    

### DATA RETRIVAL

LLM

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0 #creativity
)

llm.model_name

'openai/gpt-oss-120b'

In [41]:
test_response = llm.invoke("Hey is learning Rag hard?")
print(test_response.content)

Hey! Learning Retrieval‑Augmented Generation (RAG) can feel a bit intimidating at first, but most people find it quite manageable once they break it down into its core pieces. Here’s a quick rundown of why it might seem hard—and how to make the learning curve smoother.

---

## 1️⃣ What makes RAG feel “hard”?

| Factor | Why it can be tricky | How to tame it |
|--------|----------------------|----------------|
| **Multiple components** (retriever, vector store, generator) | You have to understand each piece and how they talk to each other. | Treat them as separate modules first; build a tiny retriever‑only demo, then a generator‑only demo, then combine. |
| **Data engineering** (embedding creation, indexing) | You need to preprocess text, choose embeddings, and manage updates. | Start with a ready‑made dataset (e.g., Wikipedia snippets) and a pre‑built vector DB like **FAISS** or **Pinecone**. |
| **Evaluation** (relevance vs. fluency) | Measuring success isn’t just “BLEU scores”; you 

### AI AGENT

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - NO MEMORY

In [42]:
from langchain.agents import create_agent

In [49]:

hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [45]:

def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [50]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [54]:
print(response)

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='151d1453-bbba-4049-84a0-f5ed82cf8908'), AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCrop**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Crop. Should answer that. No need to search policy. It\'s not about policy. So answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 208, 'total_tokens': 285, 'completion_time': 0.16015129, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.008842934, 'prompt_tokens_details': None, 'queue_time': 0.158772854, 'total_time': 0.168994224}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5a93aea882', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

System msg - HR Assisnt

Human msg - Tell me about policies

AI msg - Hey there are the policies

In [52]:
response["messages"][-2].content

'tell me which org you work for'

In [53]:
response["messages"][-1].content

'I’m the friendly HR assistant here at **Acme\u202fCrop**. How can I help you today?'